# Recipe Generator with Specific Constraints

## 7. Exercises for Practice

Now it's your turn! Complete the following exercises to practice your LangChain and prompt engineering skills.

### Exercise 1: Recipe Generator with Specific Constraints

Create a prompt that generates recipes with the following constraints:
- Must use exactly 5 ingredients (no more, no less)
- Must be cooked in 30 minutes or less
- Must be suitable for beginners
- Should include a creative title

Test your prompt with at least two different cuisine types.

# Provo qualcosa con degli input interattivi e con un pò di Magic The Gathering per i titoli creativi


## Recipe Generator con Vincoli e Tema Magic The Gathering

In questo notebook, svilupperò un generatore di ricette che rispetta vincoli specifici definiti nell'esercizio. Ho deciso di arricchire l'esperienza aggiungendo un tocco creativo: i titoli delle ricette saranno ispirati al mondo di **Magic The Gathering**, fondendo così la passione per la cucina con quella per il celebre gioco di carte.

Il generatore dovrà produrre ricette che rispettino rigorosamente questi vincoli:
- Utilizzare esattamente 5 ingredienti
- Poter essere preparate in massimo 30 minuti
- Essere adatte a cuochi principianti
- Avere un titolo creativo ispirato a Magic The Gathering

Per implementare questo progetto, utilizzerò **LangChain** con il modello **GPT-3.5 Turbo**, insieme a una struttura di **validazione Pydantic** per garantire che le ricette generate rispettino i vincoli richiesti.

In [ ]:
# !pip install ipywidgets

## Configurazione dell'Ambiente e delle Dipendenze

In questa sezione importo tutte le librerie necessarie per il nostro generatore di ricette:
- **LangChain**: per interagire con il modello linguistico e strutturare i prompt
- **Pydantic**: per definire e validare lo schema delle ricette generate
- **IPyWidgets**: per creare un'interfaccia utente interattiva all'interno del notebook
- **dotenv**: per gestire in sicurezza le chiavi API

Ho scelto di utilizzare GPT-3.5 Turbo per un buon equilibrio tra qualità delle ricette generate e velocità di risposta, impostando una temperatura di 0.7 per favorire un certo grado di creatività nei risultati.

In [1]:
# Import delle librerie
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.schema.runnable import RunnableSequence  # Import per il nuovo approccio
from langchain.output_parsers import PydanticOutputParser
from langchain.prompts.chat import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from typing import List
from pydantic import BaseModel, Field, field_validator
from ipywidgets import widgets
from IPython.display import display, HTML, clear_output

In [2]:
# carico la chiave nel mio .env
load_dotenv()

True

In [3]:
# Inizializzo il modello
chat = ChatOpenAI(
    model="gpt-3.5-turbo", 
    temperature=0.7,
    max_tokens=None,
    timeout=None,
    top_p=None,
    n=1,
    max_retries=2,
)


## Definizione dello Schema delle Ricette

Ho creato un modello di dati strutturato utilizzando Pydantic per garantire che le ricette generate rispettino tutti i vincoli dell'esercizio. La struttura include:

1. **Classe Ingredient**: definisce la struttura di ogni ingrediente con nome, quantità e unità di misura
2. **Classe RecipeStep**: rappresenta le istruzioni passo-passo per preparare la ricetta
3. **Classe Recipe**: il modello principale che contiene tutti i dettagli della ricetta

Per assicurare il rispetto dei vincoli ho implementato **tre validatori** personalizzati:
- Un validatore che controlla che ci siano esattamente 5 ingredienti
- Un validatore che verifica che il tempo di preparazione non superi i 30 minuti
- Un validatore che assicura che la difficoltà sia adatta ai principianti

Questa struttura garantisce non solo che le ricette abbiano un formato consistente, ma anche che rispettino rigorosamente i vincoli richiesti dall'esercizio.

In [4]:
# Define a Pydantic model for a recipe with the constraints
class Ingredient(BaseModel):
    name: str = Field(description="The name of the ingredient")
    quantity: str = Field(description="The quantity of the ingredient needed")
    unit: str = Field(description="The unit of measurement for the ingredient")

class RecipeStep(BaseModel):
    step_number: int = Field(description="The step number in the cooking process")
    instruction: str = Field(description="The detailed cooking instruction for this step")

class Recipe(BaseModel):
    title: str = Field(description="A creative title for the recipe")
    cooking_time: str = Field(description="Total time required to prepare and cook (must be 30 minutes or less)")
    difficulty: str = Field(description="Difficulty level (must be suitable for beginners)")
    servings: int = Field(description="Number of servings this recipe makes")
    ingredients: List[Ingredient] = Field(description="List of exactly 5 ingredients with quantities (no more, no less)")
    instructions: List[RecipeStep] = Field(description="Step-by-step cooking instructions")
    nutritional_info: str = Field(description="Brief nutritional highlights")
    serving_suggestion: str = Field(description="Suggestion on how to serve the dish")
    
    # Validator to ensure exactly 5 ingredients
    @field_validator('ingredients')
    @classmethod
    def validate_ingredients_count(cls, v):
        if len(v) != 5:
            raise ValueError('Recipe must contain exactly 5 ingredients (no more, no less)')
        return v
    
    # Validator to ensure cooking time is 30 minutes or less
    @field_validator('cooking_time')
    @classmethod
    def validate_cooking_time(cls, v):
        if "hour" in v.lower() or "hr" in v.lower():
            raise ValueError('Cooking time must be 30 minutes or less')
        try:
            # Extract numeric part of cooking time
            import re
            minutes = re.findall(r'\d+', v)
            if minutes and int(minutes[0]) > 30:
                raise ValueError('Cooking time must be 30 minutes or less')
        except:
            pass  # If we can't parse the time, we'll let it pass and rely on the prompt
        return v
    
    # Validator to ensure the recipe is suitable for beginners
    @field_validator('difficulty')
    @classmethod
    def validate_difficulty(cls, v):
        acceptable_values = ['easy', 'beginner', 'simple', 'suitable for beginners']
        if v.lower() not in acceptable_values:
            raise ValueError('Recipe must be suitable for beginners')
        return v

## Progettazione del Prompt Engineering

Il prompt è stato progettato per guidare il modello linguistico a generare ricette che rispettino tutti i vincoli richiesti. Ho strutturato il prompt in diverse sezioni:

1. **Definizione del ruolo**: Il modello assume il ruolo di uno chef professionista appassionato di Magic The Gathering
2. **Specificazione dei vincoli**: Includo esplicitamente tutti i requisiti (5 ingredienti, tempo max 30 minuti, adatto ai principianti)
3. **Requisito creativo**: Richiedo un titolo ispirato a Magic The Gathering per aggiungere originalità
4. **Parametri di input**: Il prompt accetta tre variabili:
   - Il tipo di cucina desiderato
   - Gli ingredienti disponibili
   - Eventuali restrizioni alimentari

Ho integrato anche le istruzioni di formato generate dal parser Pydantic per assicurare che l'output sia strutturato correttamente e possa essere facilmente convertito in un oggetto Recipe.

In [5]:
# Create a parser for the recipe
recipe_parser = PydanticOutputParser(pydantic_object=Recipe)

# Create a format instruction
format_instructions = recipe_parser.get_format_instructions()

# Create a structured recipe prompt with the constraints
structured_recipe_template = """You are a professional chef who specializes in creating delicious recipes that are quick and easy to make. 
You are also a huge fan of Magic The Gathering card game.

Create a {cuisine} recipe with the following constraints:
- Must use EXACTLY 5 ingredients (no more, no less)
- Must be cooked in 30 minutes or less
- Must be suitable for beginners
- IMPORTANT: The recipe MUST have a creative title inspired by a character, spell, or location from Magic The Gathering card game. 
Examples could be "Jace's Mind-Bending Pasta", "Liliana's Dark Chocolate Delight", or "Ravnica Street Tacos".

For the difficulty level, use one of these exact values: "easy", "beginner", or "simple".

Based on these requirements, create a recipe using these ingredients: {ingredients}.
Consider these dietary restrictions: {dietary_restrictions}.

{format_instructions}
"""

structured_prompt = PromptTemplate(
    template=structured_recipe_template,
    input_variables=["cuisine", "ingredients", "dietary_restrictions"],
    partial_variables={"format_instructions": format_instructions}
)

# Create the chain for structured output using the new recommended approach with parser
# We use pipe operator to compose prompt, chat model, and parser
structured_recipe_chain = structured_prompt | chat | recipe_parser

## Interfaccia Utente Interattiva

Per rendere l'esperienza più coinvolgente, ho creato un'interfaccia utente intuitiva utilizzando IPyWidgets. L'interfaccia include:

- **Menu a tendina per la cucina**: permette di selezionare tra diverse tradizioni culinarie
- **Area di testo per gli ingredienti**: consente all'utente di specificare gli ingredienti disponibili
- **Campo per restrizioni alimentari**: per indicare eventuali preferenze o allergie
- **Pulsante di generazione**: avvia il processo di creazione della ricetta
- **Pulsante di reset**: permette di ripulire tutti i campi

Questa interfaccia rende l'interazione con il generatore di ricette accessibile anche a utenti non tecnici, trasformando l'esercizio in uno strumento potenzialmente utile per chiunque voglia trovare ispirazione in cucina con ingredienti limitati.

In [6]:
# Creare i widget per l'input dell'utente
cuisine_dropdown = widgets.Dropdown(
    options=['Italian', 'Mexican', 'Japanese', 'Indian', 'French', 'Thai', 'Chinese', 'Greek'],
    value='Italian',
    description='Cuisine:',
    disabled=False,
    layout=widgets.Layout(width='300px')
)

ingredients_input = widgets.Textarea(
    value='pasta, tomatoes, garlic, basil, olive oil',
    placeholder='Enter ingredients (comma separated)',
    description='Ingredients:',
    disabled=False
)

dietary_input = widgets.Text(
    value='vegetarian',
    placeholder='Enter dietary restrictions',
    description='Dietary:',
    disabled=False
)

output_area = widgets.Output()

generate_button = widgets.Button(
    description='Generate Recipe',
    disabled=False,
    button_style='primary',  # stili disponibili 'success', 'info', 'warning', 'danger', 'primary', or ''
    tooltip='Click to generate a recipe',
    icon='utensils',  # Icona a tema cucina (FontAwesome)
    layout=widgets.Layout(width='200px', height='40px')  # Dimensioni personalizzate
)

reset_button = widgets.Button(
    description='Reset',
    button_style='danger',
    icon='trash',
    layout=widgets.Layout(width='100px', height='40px')
)


### Dopo aver definito gli input con i widgets, creo le due funzioni per i button (Reset e Generazione)

In [7]:
def on_reset_clicked(b):
    cuisine_dropdown.value = 'Italian'
    ingredients_input.value = ''
    dietary_input.value = ''
    with output_area:
        clear_output()

reset_button.on_click(on_reset_clicked)

# Funzione per il click del pulsante
def on_button_clicked(b):
    with output_area:
        clear_output()
        print("Generating recipe...")
        
        try:
            # Ottieni gli input dell'utente
            cuisine = cuisine_dropdown.value
            ingredients = ingredients_input.value
            dietary_restrictions = dietary_input.value
            
            # Genera la ricetta usando il nuovo approccio con RunnableSequence e parser
            recipe_obj = structured_recipe_chain.invoke({
                "cuisine": cuisine,
                "ingredients": ingredients,
                "dietary_restrictions": dietary_restrictions
            })
            
            # Con il parser integrato nella catena, recipe_obj è già un oggetto Recipe
            
            # Converti l'oggetto Recipe in JSON per visualizzazione
            import json
            recipe_json = recipe_obj.model_dump_json(indent=2)
            
            # Visualizza la ricetta in un formato leggibile
            html_output = f"""
            <div style='background-color: #f8f9fa; padding: 20px; border-radius: 8px; font-family: Arial, sans-serif;'>
                <h2 style='color: #d35400;'>{recipe_obj.title}</h2>
                
                <p><strong>Cooking Time:</strong> {recipe_obj.cooking_time}</p>
                <p><strong>Difficulty:</strong> {recipe_obj.difficulty}</p>
                <p><strong>Servings:</strong> {recipe_obj.servings}</p>
                
                <h3>Ingredients:</h3>
                <ul>
                {"".join([f"<li>{ing.quantity} {ing.unit} {ing.name}</li>" for ing in recipe_obj.ingredients])}
                </ul>
                
                <h3>Instructions:</h3>
                <ol>
                {"".join([f"<li>{step.instruction}</li>" for step in recipe_obj.instructions])}
                </ol>
                
                <p><strong>Nutritional Info:</strong> {recipe_obj.nutritional_info}</p>
                <p><strong>Serving Suggestion:</strong> {recipe_obj.serving_suggestion}</p>
            </div>
            
            <div style='margin-top: 20px;'>
                <details>
                    <summary style='cursor: pointer; color: #f8802a;'><strong>View JSON data</strong></summary>
                    <pre style='background-color: #f0f0f0; padding: 15px; border-radius: 5px; margin-top: 10px;'>{recipe_json}</pre>
                </details>
            </div>
            """
            display(HTML(html_output))
            
        except Exception as e:
            print(f"Si è verificato un errore: {str(e)}")
            import traceback
            traceback.print_exc()

# Collega la funzione al pulsante
generate_button.on_click(on_button_clicked)

## Presentazione dei Risultati

La visualizzazione della ricetta generata è stata curata nei dettagli per offrire un'esperienza piacevole ed informativa:

- **Intestazione con titolo creativo**: in evidenza il titolo ispirato a Magic The Gathering
- **Informazioni generali**: tempo di preparazione, difficoltà e numero di porzioni
- **Lista degli ingredienti**: formattata in modo chiaro con quantità e unità di misura
- **Istruzioni passo-passo**: numerate e dettagliate per guidare il principiante
- **Informazioni nutrizionali**: una breve panoramica dei valori nutrizionali
- **Suggerimenti di servizio**: consigli su come presentare e accompagnare il piatto

Ho aggiunto anche una sezione espandibile con i dati JSON grezzi, utile per scopi didattici o di debug.



In [8]:
# Mostra i widget nell'interfaccia
print("## 🍽️ Recipe Generator with Constraints 🍽️")
print("Complete the form below and click 'Generate Recipe' to create a recipe.")
display(cuisine_dropdown)
display(ingredients_input)
display(dietary_input)
display(widgets.HBox([generate_button, reset_button]))  # Mostro i pulsanti affiancati
display(output_area)

## 🍽️ Recipe Generator with Constraints 🍽️
Complete the form below and click 'Generate Recipe' to create a recipe.


Dropdown(description='Cuisine:', layout=Layout(width='300px'), options=('Italian', 'Mexican', 'Japanese', 'Ind…

Textarea(value='pasta, tomatoes, garlic, basil, olive oil', description='Ingredients:', placeholder='Enter ing…

Text(value='vegetarian', description='Dietary:', placeholder='Enter dietary restrictions')

Output()

## 🚨 FOrse potevo aggiungere una validazione preventiva anche lato front-end? Per evitare chiamate "in più"?.
* Potrei avere un risparmio di🚨 risorse
* Chiedere subito all'utente di sistamre il numero di ingredienti
* Evitare di avere errori post generazione

Forse potevo aggiungere qualcosa sul button tipo🚨:
```
def validate_input(ingredients, cuisine, dietary_restrictions):
    # Controlla che gli ingredienti siano esattamente 5
    ingredients_list = [i.strip() for i in ingredients.split(',') if i.strip()]
    if len(ingredients_list) != 5:
        return False, f"Please provide exactly 5 ingredients. You provided {len(ingredients_list)}."
    
    # Altri controlli...
    return True, "Input valid"

# Nel gestore del click del pulsante:
def on_button_clicked(b):
    with output_area:
        clear_output()
        
        # Ottieni gli input dell'utente
        cuisine = cuisine_dropdown.value
        ingredients = ingredients_input.value
        dietary_restrictions = dietary_input.value
        
        # Valida gli input prima di chiamare l'API
        is_valid, message = validate_input(ingredients, cuisine, dietary_restrictions)
        if not is_valid:
            print(f"Error: {message}")
            return
            
        # Procedi con la generazione...
```

## Però risulta rigido come approccio?

## 1. Selezione intelligente nel prompt

Si potrebbe modificare il prompt per indicare al modello di selezionare solo i 5 ingredienti più adatti:

structured_recipe_template = """You are a professional chef...

The user has provided these ingredients: {ingredients}.
SELECT ONLY THE 5 MOST SUITABLE INGREDIENTS from this list for your recipe.
You must use EXACTLY 5 ingredients in your recipe (no more, no less).
...
"""

## 2. Potevo fare due chiamate? una per selezionare gli ingredienti, e una per la generazione della ricetta

**Primo passaggio: seleziona gli ingredienti**
ingredients_selection_template = """Given these ingredients: {ingredients}
Select the 5 most appropriate ones for a {cuisine} dish.
List only the 5 selected ingredients, comma separated."""

**Secondo passaggio: genera la ricetta con gli ingredienti selezionati**
